#1. Carga y preparación de datos (ETL)

In [1]:
# 1.1 Librerías
import re
import sys
from pathlib import Path

_nb_dir = Path(".").resolve()
_project_root = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from config import PROCESSED_DATASET_PATH, PROCESSED_DIR

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import gaussian_kde

# Constante: nombre de columna de lectura del sensor (dominio)
COL_READING = "Leyendo"

# Patrón archivos por mes-año: nombre base + _YYYY-MM.csv
_processed_stem = PROCESSED_DATASET_PATH.stem  # ej. Procesamiento_de_Datos
_monthly_pattern = re.compile(
    "^" + re.escape(_processed_stem) + r"_(\d{4})-(\d{2})\.csv$"
)


def _load_and_clean(path):
    """Carga un CSV/Excel y aplica limpieza común (Leyendo, fechas)."""
    if str(path).endswith(".csv"):
        out = pd.read_csv(path)
    else:
        out = pd.read_excel(path)
    out[COL_READING] = pd.to_numeric(
        out[COL_READING].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    out["Tiempo de Inicio"] = pd.to_datetime(out["Tiempo de Inicio"])
    return out


# 1.2 Detectar archivos por mes-año en data/processed
monthly_dfs = []
use_fallback = False
monthly_files = []

for f in Path(PROCESSED_DIR).iterdir():
    if not f.is_file():
        continue
    m = _monthly_pattern.match(f.name)
    if m:
        monthly_files.append((int(m.group(1)), int(m.group(2)), f))

monthly_files.sort(key=lambda x: (x[0], x[1]))

for year, month, path in monthly_files:
    try:
        part = _load_and_clean(path)
        if part.empty:
            use_fallback = True
            break
        monthly_dfs.append(part)
    except Exception:
        use_fallback = True
        break

# Si no hay archivos mes-año o hubo error: usar consolidado como fallback
if not monthly_files or use_fallback:
    if str(PROCESSED_DATASET_PATH).endswith(".csv"):
        df = pd.read_csv(PROCESSED_DATASET_PATH)
    else:
        df = pd.read_excel(PROCESSED_DATASET_PATH)
    df[COL_READING] = pd.to_numeric(
        df[COL_READING].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    df["Tiempo de Inicio"] = pd.to_datetime(df["Tiempo de Inicio"])
    if use_fallback and monthly_files:
        _source = "consolidado (fallback por error en archivos mes-año)"
    else:
        _source = "consolidado (sin archivos mes-año)"
else:
    df = pd.concat(monthly_dfs, ignore_index=True)
    _source = f"{len(monthly_dfs)} archivo(s) mes-año"

# 1.3 Ordenar cronológicamente
df = df.sort_values("Tiempo de Inicio")

# Resumen de carga
print(f"Origen: {_source}. Filas: {len(df)}")
df.head()

Origen: 9 archivo(s) mes-año. Filas: 13894


,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura
0,CO,Sensor SN: 2206309128#S0001,Periodo 1,Monóxido de carbono,0.00,ppm,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:38:29,NaN,NaN,NaN,NaN,20.0
1,NO2,Sensor SN: 1807482030#S0004,Periodo 1,Dióxido de nitrógeno,0.00,ppm,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:19,NaN,NaN,NaN,NaN,20.0
2,O2,Sensor SN: 21071TQ142#S0020,Periodo 1,Oxígeno,20.40,%vol,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:29,NaN,NaN,NaN,NaN,20.0
3,LEL(CH4),Sensor SN: 1908138189#S0021,Periodo 1,Gas combustible,0.05,%vol,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:19,NaN,NaN,NaN,NaN,20.0
4,NO2,Sensor SN: 1807482030#S0004,Periodo 1,Dióxido de nitrógeno,-0.10,ppm,NaN,NaN,2023-03-31 14:37:19,2023-03-31 14:37:29,NaN,NaN,NaN,NaN,20.0


## 2.b Ventana de 24 meses, estacionalidad y aumento conservador

Este bloque prepara un dataset con **24 meses consecutivos** anclados al último mes observado. Los meses sin observaciones para cada par (Gas, Sensor) se completan por **bootstrap** de otro mes real del mismo par, remapeando marcas de tiempo (**mismas 15 columnas** que el ETL 02).

**Limitación:** con pocos meses reales no es posible demostrar estacionalidad anual; el aumento **replica** variabilidad observada, no condiciones futuras reales. Reentrenar cuando haya más datos históricos.

**Salidas:** `data/processed/Procesamiento_de_Datos_24m.csv` (misma estructura que el consolidado) y `data/processed/augmentation_manifest.json` y CSV mensuales en `data/curated/Procesamiento_de_Datos_24m_YYYY-MM.csv` (trazabilidad de bloques sintéticos). Ajusta `USAR_DATASET_24M_EN_EDA` para que el resto del notebook use o no el dataframe ampliado.


In [2]:
import json
import sys
from pathlib import Path

_nb_dir = Path(".").resolve()
_project_root = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from config import CURATED_DIR, OVERWRITE_CURATED_FILES, PROCESSED_DIR

from src.gases_augmentation import (
    audit_temporal_coverage,
    augment_to_n_months,
    assert_full_month_coverage,
    compare_reading_distributions,
    compute_target_period_range,
    validate_schema,
    write_manifest,
    write_monthly_csv_to_curated,
)

N_MONTHS = 24
USAR_DATASET_24M_EN_EDA = True
AUMENTAR_RUIDO_LECTURA = False
RNG_SEED = 42


In [3]:
validate_schema(df)
audit = audit_temporal_coverage(df)
print(json.dumps(audit, indent=2, default=str))
_start_p, _end_p = compute_target_period_range(df, n_months=N_MONTHS)
print(f"Ventana objetivo: {_start_p} .. {_end_p} ({N_MONTHS} meses)")


{
  "n_rows": 13894,
  "n_rows_with_valid_time": 13894,
  "min_time": "2023-03-31 14:37:09",
  "max_time": "2025-12-11 14:17:44",
  "periods_covered": [
    "2023-03",
    "2024-09",
    "2024-11",
    "2024-12",
    "2025-08",
    "2025-09",
    "2025-10",
    "2025-11",
    "2025-12"
  ],
  "n_distinct_months": 9,
  "rows_per_month": {
    "2023-03": 40,
    "2024-09": 513,
    "2024-11": 8937,
    "2024-12": 18,
    "2025-08": 236,
    "2025-09": 322,
    "2025-10": 184,
    "2025-11": 2344,
    "2025-12": 1300
  }
}
Ventana objetivo: 2024-01 .. 2025-12 (24 meses)


### Estacionalidad (exploratoria)

Agregación mensual por gas; subset metano/LEL; STL opcional sobre el promedio mensual global de `Leyendo`.


In [4]:
_df_audit = df.copy()
_df_audit["Tiempo de Inicio"] = pd.to_datetime(_df_audit["Tiempo de Inicio"], errors="coerce")
_df_audit["_mes"] = _df_audit["Tiempo de Inicio"].dt.to_period("M").astype(str)
monthly_gas = (
    _df_audit.dropna(subset=["Tiempo de Inicio", "Leyendo"])
    .groupby(["_mes", "Gas"], as_index=False)["Leyendo"]
    .mean()
)
fig_m = px.line(
    monthly_gas,
    x="_mes",
    y="Leyendo",
    color="Gas",
    title="Media mensual de Leyendo por gas (datos cargados)",
)
fig_m.show()

_metano = _df_audit[
    (_df_audit["Gas"].astype(str).str.contains("combustible", case=False, na=False))
    | (_df_audit["Descripción"].astype(str).str.upper().str.contains("LEL", na=False))
].copy()
if not _metano.empty:
    mt = (
        _metano.dropna(subset=["Tiempo de Inicio", "Leyendo"])
        .groupby(["_mes", "Gas"], as_index=False)["Leyendo"]
        .mean()
    )
    fig_ch4 = px.line(
        mt,
        x="_mes",
        y="Leyendo",
        color="Gas",
        title="Serie mensual — foco metano / LEL",
    )
    fig_ch4.show()
else:
    print("Sin filas para subset metano/LEL.")


In [5]:
try:
    from statsmodels.tsa.seasonal import STL
except ImportError:
    STL = None

if STL is None:
    print("Instala statsmodels para STL: pip install statsmodels")
else:
    g = _df_audit.dropna(subset=["Tiempo de Inicio", "Leyendo"]).copy()
    g["_mes_ts"] = g["Tiempo de Inicio"].dt.to_period("M").dt.to_timestamp()
    _ts = g.groupby("_mes_ts")["Leyendo"].mean().sort_index()
    n = len(_ts)
    if n < 6:
        print("Pocos meses para STL fiable; agregar más datos o omitir.")
    else:
        seasonal = 7 if n > 24 else (5 if n > 14 else 3)
        if n <= 2 * seasonal:
            print(f"STL omitido: n={n} no supera 2*seasonal ({2 * seasonal}).")
        else:
            res = STL(_ts, seasonal=seasonal, robust=True).fit()
            fig_stl = go.Figure()
            fig_stl.add_trace(go.Scatter(x=_ts.index, y=res.seasonal, name="Estacional"))
            fig_stl.update_layout(
                title="STL — componente estacional (promedio mensual global Leyendo)",
            )
            fig_stl.show()


Instala statsmodels para STL: pip install statsmodels


### Aumentación, validación y guardado (24 meses)


In [6]:
from IPython.display import display

_df_before = df.copy()
df_augmented, manifest_entries = augment_to_n_months(
    df,
    n_months=N_MONTHS,
    seed=RNG_SEED,
    apply_reading_noise=AUMENTAR_RUIDO_LECTURA,
)

cov = assert_full_month_coverage(df_augmented, n_months=N_MONTHS)
print(json.dumps(cov, indent=2, default=str))

_cmp = compare_reading_distributions(_df_before, df_augmented)
display(_cmp)

_out_csv = PROCESSED_DIR / "Procesamiento_de_Datos_24m.csv"
_out_manifest = PROCESSED_DIR / "augmentation_manifest.json"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_augmented.to_csv(_out_csv, index=False, encoding="utf-8-sig")
write_manifest(
    _out_manifest,
    manifest_entries,
    meta={
        "n_months": N_MONTHS,
        "seed": RNG_SEED,
        "reading_noise": AUMENTAR_RUIDO_LECTURA,
        "output_csv": str(_out_csv.name),
    },
)
print(f"Guardado: {_out_csv} ({len(df_augmented)} filas)")
print(f"Manifest: {_out_manifest} ({len(manifest_entries)} bloques sintéticos)")

_curated_report = write_monthly_csv_to_curated(
    df_augmented,
    CURATED_DIR,
    filename_stem="Procesamiento_de_Datos_24m",
    overwrite=OVERWRITE_CURATED_FILES,
)
print(json.dumps(_curated_report, indent=2, default=str))
print(f"Data final mes a mes (curated): {CURATED_DIR}")

if USAR_DATASET_24M_EN_EDA:
    df = df_augmented


{
  "start_period": "2024-01",
  "end_period": "2025-12",
  "months_present": 24,
  "months_expected": 24,
  "missing_periods": [],
  "is_complete": true
}


,Gas,n_before,mean_before,std_before,n_after,mean_after,std_after
0,Dióxido de carbono,4698,0.489977,0.908723,60891,0.463156,0.885343
1,Dióxido de nitrógeno,748,0.467513,1.723505,10280,0.427510,1.615412
2,Gas combustible,491,0.551752,4.966481,6298,0.957259,6.760815
3,Monóxido de carbono,2716,7.504786,45.315464,28180,7.503797,44.718136
4,Oxígeno,2694,19.759465,2.222973,38530,19.748412,2.600443
5,Sulfuro de hidrógeno,2547,0.818924,2.585439,32559,0.810590,2.781308


Guardado: C:\Users\FA-BI\OneDrive\Documentos\Maestria IA\Tesis\proyecto-gases\data\processed\Procesamiento_de_Datos_24m.csv (176738 filas)
Manifest: C:\Users\FA-BI\OneDrive\Documentos\Maestria IA\Tesis\proyecto-gases\data\processed\augmentation_manifest.json (1136 bloques sintéticos)


#2. EDA – Análisis estadístico descriptivo

In [7]:
df["Leyendo"].describe()

count    176738.000000
mean          5.869598
std          19.584967
min         -21.000000
25%           0.000000
50%           0.200000
75%           5.000000
max         350.000000
Name: Leyendo, dtype: float64

2.1 Estadísticos generales

In [8]:
df["Leyendo"].describe()

count    176738.000000
mean          5.869598
std          19.584967
min         -21.000000
25%           0.000000
50%           0.200000
75%           5.000000
max         350.000000
Name: Leyendo, dtype: float64

2.2 Estadísticos por gas

In [9]:
df.groupby("Gas")["Leyendo"].describe()

,count,mean,std,min,25%,50%,75%,max
Gas,,,,,,,,
Dióxido de carbono,60891.0,0.463156,0.885343,-0.17,0.010000,0.08,0.480000,5.0
Dióxido de nitrógeno,10280.0,0.427510,1.615412,-0.90,-0.400000,0.00,0.300000,8.6
Gas combustible,6298.0,0.957259,6.760815,-0.14,0.000000,0.10,0.120000,61.0
Monóxido de carbono,28180.0,7.503797,44.718136,-21.00,-5.000000,0.00,3.000000,350.0
Oxígeno,38530.0,19.748412,2.600443,0.00,19.700001,20.40,20.700001,30.0
Sulfuro de hidrógeno,32559.0,0.810590,2.781308,-1.60,0.000000,0.20,0.600000,24.1


2.3 Conteo de registros por gas

In [10]:
df["Gas"].value_counts()

Gas
Dióxido de carbono      60891
Oxígeno                 38530
Sulfuro de hidrógeno    32559
Monóxido de carbono     28180
Dióxido de nitrógeno    10280
Gas combustible          6298
Name: count, dtype: int64

#3️. EDA – Análisis de distribuciones

3.1 Histogramas por gas

In [11]:
gases = df["Gas"].unique()

for gas in gases:
    subset = df[df["Gas"] == gas]["Leyendo"].dropna()

    if len(subset) > 1:
        kde = gaussian_kde(subset)
        x_range = np.linspace(subset.min(), subset.max(), 200)

        fig = go.Figure()

        fig.add_trace(go.Histogram(
            x=subset,
            nbinsx=30,
            histnorm="density",
            name="Frecuencia",
            marker_color="steelblue",
            opacity=0.6
        ))

        fig.add_trace(go.Scatter(
            x=x_range,
            y=kde(x_range),
            mode="lines",
            name="KDE",
            line=dict(color="crimson", width=2)
        ))

        fig.update_layout(
            title=f"Distribución de concentración — {gas}",
            xaxis_title="Concentración",
            yaxis_title="Densidad",
            template="plotly_white",
            hovermode="x unified"
        )

        fig.show()

3.2 Boxplot interactivo

In [12]:
fig_box = px.box(
    df,
    x="Gas",
    y="Leyendo",
    color="Gas",
    template="plotly_white",
    title="Detección de Outliers por Gas"
)

fig_box.update_layout(showlegend=False)
fig_box.show()

#4. Análisis Temporal

4.1 Serie temporal por gas

In [13]:
for gas in gases:
    subset = df[df["Gas"] == gas]

    fig = px.line(
        subset,
        x="Tiempo de Inicio",
        y="Leyendo",
        title=f"Serie Temporal — {gas}",
        template="plotly_white"
    )

    fig.show()

4.2 Tendencia con media móvil

In [14]:
df["Media_Movil"] = df["Leyendo"].rolling(window=20).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["Tiempo de Inicio"],
    y=df["Leyendo"],
    mode="lines",
    name="Valor original",
    line=dict(color="lightgray")
))

fig.add_trace(go.Scatter(
    x=df["Tiempo de Inicio"],
    y=df["Media_Movil"],
    mode="lines",
    name="Media móvil",
    line=dict(color="red", width=2)
))

fig.update_layout(
    title="Tendencia con Media Móvil",
    template="plotly_white"
)

fig.show()

4.3 Detección de cambios abruptos

In [15]:
df["Cambio"] = df["Leyendo"].diff().abs()

threshold = df["Cambio"].mean() + 2 * df["Cambio"].std()
df["Cambio_Brusco"] = df["Cambio"] > threshold

df[df["Cambio_Brusco"] == True].head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco
6318,CO,Sensor SN: 22061PG059#S0001,Periodo 2,Monóxido de carbono,71.0,ppm,"0,79","0,02",2024-01-27 10:44:01,2024-01-27 10:44:11,NaN,NaN,NaN,Alta,30.0,3.3395,70.990000,True
6322,CO,Sensor SN: 22061PG059#S0001,Periodo 2,Monóxido de carbono,98.0,ppm,"1,88","0,06",2024-01-27 10:44:11,2024-01-27 10:44:21,NaN,NaN,NaN,Alta,30.0,10.4620,83.400000,True
6323,H2S,Sensor SN: 22061PG059#S0002,Periodo 2,Sulfuro de hidrógeno,21.1,ppm,"0,40","0,01",2024-01-27 10:44:11,2024-01-27 10:44:21,NaN,NaN,NaN,Alta,30.0,11.5470,76.900000,True
6325,LEL,Sensor SN: 22081FT036#S0022,Periodo 2,Gas combustible,59.0,%lel,NaN,NaN,2024-01-27 10:44:11,2024-01-27 10:44:21,NaN,NaN,Pentano,Alta,30.0,14.6345,57.250000,True
6328,CO,Sensor SN: 22061PG059#S0001,Periodo 2,Monóxido de carbono,101.0,ppm,"5,24","0,16",2024-01-27 10:44:21,2024-01-27 10:44:51,NaN,NaN,NaN,Alta,30.0,23.9335,78.799999,True


#5. EDA – Análisis multivariable

5.1 Pivotear datos por gas

In [16]:
df_pivot = df.pivot_table(
    index="Tiempo de Inicio",
    columns="Gas",
    values="Leyendo"
)

5.2 Matriz de correlación interactiva

In [17]:
fig_corr = px.imshow(
    df_pivot.corr(),
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Matriz de Correlación entre Gases"
)

fig_corr.show()

#6. EDA – Patrones previos a eventos

6.1 Definir evento (Percentil 95)

In [18]:
limite = df["Leyendo"].quantile(0.95)
eventos = df[df["Leyendo"] > limite]

eventos.head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco
3,O2,Sensor SN: 241033Q134#S0020,Periodo 2,Oxígeno,21.0,%vol,NaN,NaN,2024-01-03 06:01:48,2024-01-03 06:03:48,NaN,NaN,NaN,NaN,12.0,NaN,21.00,False
12,O2,Sensor SN: 241033Q134#S0020,Periodo 2,Oxígeno,21.0,%vol,NaN,NaN,2024-01-03 06:08:48,2024-01-03 06:12:48,NaN,NaN,NaN,NaN,13.0,NaN,21.13,False
143,O2,Sensor SN: 241033Q134#S0020,Periodo 2,Oxígeno,21.0,%vol,NaN,NaN,2024-01-03 07:10:48,2024-01-03 07:13:48,NaN,NaN,NaN,NaN,26.0,5.2315,21.00,False
148,O2,Sensor SN: 241033Q134#S0020,Periodo 2,Oxígeno,21.1,%vol,NaN,NaN,2024-01-03 07:13:48,2024-01-03 07:16:48,NaN,NaN,NaN,NaN,26.0,4.1625,21.20,False
152,O2,Sensor SN: 241033Q134#S0020,Periodo 2,Oxígeno,21.0,%vol,NaN,NaN,2024-01-03 07:16:48,2024-01-03 07:32:48,NaN,NaN,NaN,NaN,25.0,5.8130,21.10,False


6.2 Analizar ventana previa de 30 minutos

In [19]:
ventanas = []

for tiempo in eventos["Tiempo de Inicio"]:
    ventana = df[
        (df["Tiempo de Inicio"] >= tiempo - pd.Timedelta(minutes=30)) &
        (df["Tiempo de Inicio"] <= tiempo)
    ]
    ventanas.append(ventana)

df_ventanas = pd.concat(ventanas)

df_ventanas.head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco
0,CO,Sensor SN: 2209366026#S0001,Periodo 2,Monóxido de carbono,0.00,ppm,NaN,NaN,2024-01-03 06:01:48,2024-01-03 06:03:48,NaN,NaN,NaN,NaN,12.0,NaN,NaN,False
1,CO2,Sensor SN: 2209364017#S0011,Periodo 2,Dióxido de carbono,-0.13,%vol,NaN,NaN,2024-01-03 06:01:48,2024-01-03 06:07:48,NaN,NaN,NaN,NaN,12.0,NaN,0.13,False
2,LEL(CH4),Sensor SN: 22082AR052#S0021,Periodo 2,Gas combustible,0.00,%vol,NaN,NaN,2024-01-03 06:01:48,2024-01-03 06:24:48,NaN,NaN,Metano,NaN,12.0,NaN,0.13,False
3,O2,Sensor SN: 241033Q134#S0020,Periodo 2,Oxígeno,21.00,%vol,NaN,NaN,2024-01-03 06:01:48,2024-01-03 06:03:48,NaN,NaN,NaN,NaN,12.0,NaN,21.00,False
4,NO2,Sensor SN: 23081F0018#S0004,Periodo 2,Dióxido de nitrógeno,0.00,ppm,NaN,NaN,2024-01-03 06:01:48,2024-01-03 06:24:48,NaN,NaN,NaN,NaN,12.0,NaN,21.00,False
